# Langfuse Demo — Chapter 3: Prompt Management + Sessions

> **Estimated time**: ~10 minutes

---

## Two classic problems we will solve

### Problem 1: Prompt Management

You improve the assistant's system prompt. How do you know if it actually improved?
- You need to re-run the same tests with v1 and v2
- Compare scores side by side
- Easy rollback if it gets worse

**Langfuse Prompt Management** = Git for prompts. Versioned, audited, decoupled from code.

### Problem 2: Multi-turn Conversations

A chatbot generates dozens of traces per user.  
How do you group the traces from the same conversation?

**Sessions** = groups all traces from a conversation under a `session_id`.  
In the UI: **Session Explorer** shows the full conversation, trace by trace.

---

In [1]:
from langfuse import propagate_attributes
from langfuse_demo.client import get_client
from langfuse_demo.config import settings
from langfuse_demo.llm import call_llm
from langfuse_demo.eval import evaluate_with_llm

from dotenv import load_dotenv

_ = load_dotenv()

langfuse = get_client()
print(f"Langfuse: {settings.langfuse_host}")

Langfuse: http://localhost:3000


---

## Part 1: Prompt Management

### Prompt v1 — Initial version (basic)

In [8]:
PROMPT_NAME = "docs-assistant"

# Creates prompt v1 in Langfuse (or updates if it already exists)
langfuse.create_prompt(
    name=PROMPT_NAME,
    type="chat",
    prompt=[
        {
            "role": "system",
            "content": "You are a technical assistant. Answer questions using the provided context.",
        },
        {
            "role": "user",
            "content": "Context:\n{{context}}\n\nQuestion: {{question}}",
        },
    ],
    labels=["staging"],
    config={"model": settings.groq_model, "max_tokens": 512},
)

print(f"Prompt v1 created at: {settings.langfuse_host}/prompts")

Prompt v1 created at: http://localhost:3000/prompts


In [9]:
TEST_CONTEXT = (
    "To set up the environment: install Python 3.12, uv, Docker. "
    "Clone the repository, run 'uv sync' and 'docker compose up -d'. "
    "Configure .env by copying from .env.example."
)
TEST_QUESTION = "What are the steps to set up the development environment?"

prompt_v1 = langfuse.get_prompt(PROMPT_NAME, label="staging", type="chat")
messages_v1 = prompt_v1.compile(context=TEST_CONTEXT, question=TEST_QUESTION)

with langfuse.start_as_current_observation(
    as_type="generation",
    name="answer-v1",
    input=messages_v1,
    prompt=prompt_v1,
) as gen:
    answer_v1, usage = call_llm(messages_v1)
    gen.update(
        model=settings.groq_model,
        output=answer_v1,
        usage_details={"input": usage["input"], "output": usage["output"]},
        cost_details={"input": usage["input_cost"], "output": usage["output_cost"]},
    )
    trace_id_v1 = langfuse.get_current_trace_id()

langfuse.flush()
print(f"Answer v1:\n{answer_v1}")
print(f"\nTokens: {usage['input']} input | {usage['output']} output")

Answer v1:
To set up the development environment, follow these steps:

1. **Install required software**: Install Python 3.12, uv, and Docker.
2. **Clone the repository**: Clone the project repository to your local machine.
3. **Run 'uv sync'**: Execute the command 'uv sync' to synchronize the project.
4. **Start Docker containers**: Run 'docker compose up -d' to start the Docker containers in detached mode.
5. **Configure .env file**: Copy the contents from the .env.example file and paste them into a new .env file to configure the environment variables.

By completing these steps, you should have a functional development environment set up.

Tokens: 107 input | 139 output


In [10]:
# Evaluates v1 answer with LLM-judge
score_v1, comment_v1 = evaluate_with_llm(TEST_QUESTION, answer_v1)

langfuse.create_score(
    trace_id=trace_id_v1,
    name="llm-judge-quality",
    value=score_v1,
    data_type="NUMERIC",
    comment=f"[v1] {comment_v1}",
)
langfuse.flush()

print(f"Score v1: {score_v1:.2f} — {comment_v1}")

Score v1: 0.90 — clear steps but lacks specific details for troubleshooting or handling potential errors


### Prompt v2 — Improved version

Improvements: more specific instruction on format, tone, and context constraint.

In [11]:
PROMPT_NAME_V2 = "docs-assistant"

langfuse.create_prompt(
    name=PROMPT_NAME_V2,
    type="chat",
    prompt=[
        {
            "role": "system",
            "content": (
                "You are a senior technical assistant for the Data Science team. "
                "Answer ONLY based on the provided context, clearly and directly. "
                "Use numbered lists for sequential steps. "
                "If the information is not in the context, say: 'I could not find that information in the documentation.' "
                "Never invent information."
            ),
        },
        {
            "role": "user",
            "content": "Documentation:\n{{context}}\n\nQuestion: {{question}}",
        },
    ],
    labels=["production"],
    config={"model": settings.groq_model, "max_tokens": 512},
)

print(f"Prompt v2 created (label: production)")
print(f"View comparison: {settings.langfuse_host}/prompts")

Prompt v2 created (label: production)
View comparison: http://localhost:3000/prompts


In [6]:
prompt_v2 = langfuse.get_prompt(PROMPT_NAME_V2, label="production", type="chat")
messages_v2 = prompt_v2.compile(context=TEST_CONTEXT, question=TEST_QUESTION)

with langfuse.start_as_current_observation(
    as_type="generation",
    name="answer-v2",
    input=messages_v2,
    prompt=prompt_v2,
) as gen:
    answer_v2, usage2 = call_llm(messages_v2)
    gen.update(
        model=settings.groq_model,
        output=answer_v2,
        usage_details={"input": usage2["input"], "output": usage2["output"]},
        cost_details={"input": usage2["input_cost"], "output": usage2["output_cost"]},
    )
    trace_id_v2 = langfuse.get_current_trace_id()

langfuse.flush()
print(f"Answer v2:\n{answer_v2}")

Answer v2:
Here are the steps to set up the environment:
1. Install Python 3.12, uv, and Docker.
2. Clone the repository.
3. Run 'uv sync'.
4. Run 'docker compose up -d'.
5. Configure .env by copying from .env.example.


In [7]:
score_v2, comment_v2 = evaluate_with_llm(TEST_QUESTION, answer_v2)

langfuse.create_score(
    trace_id=trace_id_v2,
    name="llm-judge-quality",
    value=score_v2,
    data_type="NUMERIC",
    comment=f"[v2] {comment_v2}",
)
langfuse.flush()

print(f"Score v2: {score_v2:.2f} — {comment_v2}")
print()
print("=" * 50)
winner = "v2 WON" if score_v2 > score_v1 else ("TIE" if score_v2 == score_v1 else "v1 WON")
print(f"v1: {score_v1:.2f}  |  v2: {score_v2:.2f}  →  {winner}")
print("=" * 50)
print(f"\nIn the UI (Scores): filter by name 'llm-judge-quality' and tag 'prompt-ab'")
print(f"to compare v1 vs v2 across multiple questions.")

Score v2: 0.90 — clear step-by-step guide but lacks detailed explanations and potential troubleshooting tips

v1: 0.90  |  v2: 0.90  →  TIE

In the UI (Scores): filter by name 'llm-judge-quality' and tag 'prompt-ab'
to compare v1 vs v2 across multiple questions.


---

## Part 2: Sessions — Multi-turn Conversations

A `session_id` groups multiple traces from the same conversation.  
In the UI: **Sessions** → click any session → see all turns in order.

Scenario: a new team member asks about onboarding in 3 turns.

In [12]:
import time
from langfuse_demo.rag import retrieve, rerank, format_context

SESSION_ID = f"onboarding-pedro-{int(time.time())}"
USER_ID = "pedro.lima"

conversation = [
    "I'm new to the team. Where do I start to set up my environment?",
    "What are the system prerequisites I need to install?",
    "What should I do if port 5432 is already in use?",
]

print(f"Session ID: {SESSION_ID}")
print(f"Simulating onboarding conversation with {len(conversation)} turns...\n")

Session ID: onboarding-pedro-1782813492
Simulating onboarding conversation with 3 turns...



In [13]:
trace_ids = []

for i, question in enumerate(conversation, 1):
    print(f"Turn {i}: {question}")

    docs = retrieve(question, top_k=2)
    docs = rerank(question, docs)
    context = format_context(docs, max_chars=800)

    messages = [
        {
            "role": "system",
            "content": (
                "You are an onboarding assistant for the team. "
                "Answer in a friendly and direct way. "
                "Use the provided context."
            ),
        },
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]

    # propagate_attributes: user_id, session_id and tags go to the entire trace
    with propagate_attributes(
        user_id=USER_ID,
        session_id=SESSION_ID,
        tags=["onboarding", "multi-turn", "ch-03"],
    ):
        with langfuse.start_as_current_observation(
            as_type="span",
            name=f"turn-{i}",
            input={"question": question},
            metadata={"turn": i, "context_docs": [d["id"] for d in docs]},
        ) as span:
            trace_id = langfuse.get_current_trace_id()

            with langfuse.start_as_current_observation(
                as_type="generation",
                name="llm-answer",
                input=messages,
            ) as gen:
                answer, usage = call_llm(messages)
                gen.update(
                    model=settings.groq_model,
                    output=answer,
                    usage_details={"input": usage["input"], "output": usage["output"]},
                    cost_details={"input": usage["input_cost"], "output": usage["output_cost"]},
                )

            span.update(output={"answer": answer})
            trace_ids.append(trace_id)

    print(f"  → {answer[:120]}..." if len(answer) > 120 else f"  → {answer}")
    print(f"  Tokens: {usage['input']} input | {usage['output']} output\n")

langfuse.flush()

print("=" * 60)
print(f"Session: {SESSION_ID}")
print(f"User   : {USER_ID}")
print(f"Traces : {len(trace_ids)} created")
print(f"\n🔗 View session: {settings.langfuse_host}/sessions/{SESSION_ID}")

Turn 1: I'm new to the team. Where do I start to set up my environment?
  → Welcome to the team. To set up your local development environment, start by checking the prerequisites: make sure you ha...
  Tokens: 232 input | 92 output

Turn 2: What are the system prerequisites I need to install?
  → To set up the local development environment, you'll need to have the following prerequisites installed:

1. Python 3.12 ...
  Tokens: 225 input | 50 output

Turn 3: What should I do if port 5432 is already in use?
  → If port 5432 is already in use, you should change the `POSTGRES_PORT` in the `docker-compose.yaml` file. This will allow...
  Tokens: 229 input | 43 output

Session: onboarding-pedro-1782813492
User   : pedro.lima
Traces : 3 created

🔗 View session: http://localhost:3000/sessions/onboarding-pedro-1782813492


---

## What to explore in the UI

### Prompt Management
- **Prompts** → see `docs-assistant-v1` and `docs-assistant-v2`
- Click any prompt → see version history
- **Scores** → filter by `llm-judge-quality` + tag `prompt-ab` → compare v1 vs v2

### Sessions
- **Sessions** → find the session by the ID printed above
- See the 3 turns in chronological order
- Filter traces by `user_id = pedro.lima` → see all activity for the user

---

## Chapter 3 Summary

| Feature | API |
|---------|-----|
| Create versioned prompt | `langfuse.create_prompt(name, type, prompt, labels)` |
| Fetch production version | `langfuse.get_prompt(name, label="production", type="chat")` |
| Compile variables | `prompt.compile(variable=value)` |
| Link generation to prompt | `start_as_current_observation(prompt=prompt_obj)` |
| Group conversation | `session_id=SESSION_ID` in each trace |
| Track by user | `user_id=USER_ID` in each trace |

**Next chapter**: Evaluation at scale with Datasets + LLM-judge in batch